In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [3]:
import polars as pl
import altair as alt

# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsforecast import StatsForecast
# import calendar
from hijridate import Gregorian
from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
# from scipy.stats import pearsonr
# from plotly import express as px
# from pathlib import Path
# from itertools import chain


import importlib
import lib
import read_data
importlib.reload(lib)

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(10)

polars.config.Config

In [4]:
cabs_inter = set(read_data.read_smi_envoisbyproduitintern_many()["codeenvoi_"])

In [5]:
fields = (
    pl.read_parquet("data/cab_dfs/fields.parquet")
    .filter(pl.col("Cab").is_in(cabs_inter))
    .filter(pl.col("Date_depot").ge(pl.date(2023, 1, 1)))
    .sort("Date_depot")
    .rename({"Poids_global_en_KG": "weight_kg"})
    .drop_nulls("weight_kg")
)
operations = pl.read_parquet("data/cab_dfs/operations.parquet").filter(
    pl.col("cab").is_in(cabs_inter)
)
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet").filter(
    pl.col("cab").is_in(cabs_inter)
)
services = pl.read_parquet("data/cab_dfs/services.parquet").filter(
    pl.col("cab").is_in(cabs_inter)
)

In [6]:
print(fields.head())

shape: (5, 21)
┌───────────────┬──────────┬────────────┬──────────┬────────────────┬────────┬─────────┬──────┬───────────┬───────────────────────────────────────────────────┬─────────────┬───────────────┬────────────────────────┬───────────────────────────────────────────────────────┬──────────────┬──────────────────────┬──────────────────────┬──────────┬─────────┬─────────┬────────────────────┐
│ Cab           ┆ Id       ┆ Date_depot ┆ Type_cab ┆ Dernier_statut ┆ Regime ┆ Contrat ┆ Etat ┆ weight_kg ┆ Centre_Agence_depot                               ┆ Destination ┆ Client        ┆ Produit_Niveau_Service ┆ Mode_paiement                                         ┆ Taxe_DTQ_Dhs ┆ Canal_de_livraison_1 ┆ Canal_de_livraison_2 ┆ Longueur ┆ Hauteur ┆ Largeur ┆ Poids_Volumetrique │
│ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---            ┆ ---    ┆ ---     ┆ ---  ┆ ---       ┆ ---                                               ┆ ---         ┆ ---           ┆ ---                    ┆ --

In [7]:
print(fields.select(pl.all().n_unique()))

shape: (1, 21)
┌────────┬────────┬────────────┬──────────┬────────────────┬────────┬─────────┬──────┬───────────┬─────────────────────┬─────────────┬────────┬────────────────────────┬───────────────┬──────────────┬──────────────────────┬──────────────────────┬──────────┬─────────┬─────────┬────────────────────┐
│ Cab    ┆ Id     ┆ Date_depot ┆ Type_cab ┆ Dernier_statut ┆ Regime ┆ Contrat ┆ Etat ┆ weight_kg ┆ Centre_Agence_depot ┆ Destination ┆ Client ┆ Produit_Niveau_Service ┆ Mode_paiement ┆ Taxe_DTQ_Dhs ┆ Canal_de_livraison_1 ┆ Canal_de_livraison_2 ┆ Longueur ┆ Hauteur ┆ Largeur ┆ Poids_Volumetrique │
│ ---    ┆ ---    ┆ ---        ┆ ---      ┆ ---            ┆ ---    ┆ ---     ┆ ---  ┆ ---       ┆ ---                 ┆ ---         ┆ ---    ┆ ---                    ┆ ---           ┆ ---          ┆ ---                  ┆ ---                  ┆ ---      ┆ ---     ┆ ---     ┆ ---                │
│ u32    ┆ u32    ┆ u32        ┆ u32      ┆ u32            ┆ u32    ┆ u32     ┆ u32  ┆ u32 

In [16]:
df = (
    fields.pipe(lib.aggregate_by_date, "7d", "weight_kg", "Date_depot")
    .with_columns(
        hijri_year=pl.col("Date_depot").map_elements(
            lambda dt: Gregorian.fromdate(dt).to_hijri().year
        ),
        hijri_month=pl.col("Date_depot").map_elements(
            lambda dt: Gregorian.fromdate(dt).to_hijri().month
        ),
    )
    .with_columns(
        is_ramadan=pl.col("hijri_month") == 9,
    )
    .with_columns(
        (pl.col("is_ramadan").ne(pl.col("is_ramadan").shift()).cum_sum()).alias(
            "segment"
        )
    )
)

chart = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x="Date_depot:T",
        y=alt.Y("mean:Q"),
        color="is_ramadan:N",
        detail="segment:N",
    )
    .properties(width=1100)
)
chart

alt.Chart(...)

In [ ]:
df = (
    fields.sort("weight_kg")
    .with_columns(
        pl.col("weight_kg")
        .cut(
            [float(x) for x in range(int(fields["weight_kg"].max()))],
            left_closed=True,
        )
        .alias("bucket")
    )
    .group_by("bucket", maintain_order=True)
    .agg(
        pl.col("weight_kg").mean().alias("weight_kg"),
        pl.col("weight_kg").sum().alias("weight_sum"),
    )
    .with_columns(cum_weight_sum=pl.col("weight_sum").cum_sum(reverse=True))
)

base = alt.Chart(df).properties(width=1100).mark_line(point=True)

sum_chart = base.encode(
    x=("bucket"),
    y="weight_sum",
    tooltip=[
        "bucket",
        "weight_sum",
    ],
)

cum_sum_chart = base.encode(
    x="bucket",
    y="cum_weight_sum",
    tooltip=[
        "bucket",
        "cum_weight_sum",
    ],
)
chart = sum_chart & cum_sum_chart

chart.resolve_legend()

alt.VConcatChart(...)